# V10 contextual notice checker
Frozen local pretrained-model prototype. It compares complete contexts and explicit bilingual facts, with a sentence-by-sentence translation cross-check. No V7 fallback is used. A clean result means no discrepancy was found, not that a translation is certified.
The NLLB component is a research model under CC-BY-NC-4.0; see the project README and model card for deployment limitations.


In [ ]:
!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 sentencepiece pandas scipy opencc-python-reimplemented==0.1.7


In [ ]:
from pathlib import Path
import base64, json, sys, hashlib
folder=Path('/content/v10')
folder.mkdir(exist_ok=True)
payload={'engine.py': 'IiIiVjEwOiBkaXJlY3QgY29udGV4dHVhbCBpbmZlcmVuY2UgYW5kIGJpbGluZ3VhbCBmYWN0cywgd2l0aCBubyBWNyBkZXBlbmRlbmN5LgoKVGhlIENoaW5lc2UtdG8tRW5nbGlzaCB0cmFuc2xhdGlvbiBpcyBhIGNyb3NzLWNoZWNrLCBuZXZlciBhIHJlZmVyZW5jZSBhbnN3ZXIuClNjb3JlcyByZW1haW4gZGlhZ25vc3RpYyBtb2RlbCBzY29yZXMsIG5vdCBjYWxpYnJhdGVkIHByb2JhYmlsaXRpZXMgb2YgZXJyb3IuCiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBkaWZmbGliCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tlbml6ZXIsIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24sIEF1dG9Nb2RlbEZvclNlcTJTZXFMTQpmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgpmcm9tIHNjaXB5Lm9wdGltaXplIGltcG9ydCBsaW5lYXJfc3VtX2Fzc2lnbm1lbnQKZnJvbSBmYWN0cyBpbXBvcnQgbm9ybWFsLCBkaXJlY3RfZXZpZGVuY2UKZnJvbSB0ZXJtaW5vbG9neSBpbXBvcnQgdHJhbnNsYXRpb25fc291cmNlCgpNT0RFTF9SRVZJU0lPTlMgPSBkaWN0KGVtYmVkZGluZz0nODM2MTIxYTA1MzNlNTY2NGIyMWM3YWFjYzVkMjI5NTFmMmI4YjI1YicsbmxpPSdiNTExM2ViMzhhYjYzZWZkZDdmMjgwZjhjMTQ0ZWE4YjEzZjk3OGNlJyx0cmFuc2xhdGlvbj0nZjhkMzMzYTA5OGQxOWI0ZmQ5YThiMThmOTQxNzA0ODdhZDNmODIxZCcpCgpNT0RFTF9JRFMgPSBkaWN0KGVtYmVkZGluZz0nc2VudGVuY2UtdHJhbnNmb3JtZXJzL0xhQlNFJywKICAgIG5saT0nTW9yaXR6TGF1cmVyL21EZUJFUlRhLXYzLWJhc2UteG5saS1tdWx0aWxpbmd1YWwtbmxpLTJtaWw3JywKICAgIHRyYW5zbGF0aW9uPSdmYWNlYm9vay9ubGxiLTIwMC1kaXN0aWxsZWQtNjAwTScpCgoKZGVmIHNlbnRlbmNlcyh0ZXh0KToKICAgICMgS2VlcCBzZW1pY29sb25zIGFuZCBjb21tYXMgd2l0aGluIHByb3Bvc2l0aW9uczsgcHJlc2VydmUgcmVvcmRlcmluZ3MuCiAgICByZXR1cm4gW3guc3RyaXAoKSBmb3IgeCBpbiByZS5zcGxpdChyJyg/PD1b44CC77yB77yfXSl8KD88PVsuIT9dKVxzKyg/PVtBLVpdKXxcbisnLCB0ZXh0KSBpZiB4LnN0cmlwKCldCgoKZGVmIGFsaWduZWRfZ3JvdXBzKHpzLCBlcywgZW1iZWRkaW5ncyk6CiAgICBpZiBub3QgenMgb3Igbm90IGVzOnJldHVybiBbXQogICAgc2ltPW5wLmFycmF5KFtbZmxvYXQoZW1iZWRkaW5nc1t6XUBlbWJlZGRpbmdzW2VdKSBmb3IgZSBpbiBlc10gZm9yIHogaW4genNdKQogICAgaWksamo9bGluZWFyX3N1bV9hc3NpZ25tZW50KC1zaW0pCiAgICBncm91cHM9WyhbaW50KGkpXSxbaW50KGopXSxmbG9hdChzaW1baSxqXSkpIGZvciBpLGogaW4gemlwKGlpLGpqKV0KICAgICMgUHJlc2VydmUgdW5tYXRjaGVkIG1hdGVyaWFsIGJ5IGF0dGFjaGluZyBpdCB0byB0aGUgY2xvc2VzdCBtYXRjaGVkIGdyb3VwLgogICAgIyBUaGUgZnVsbC1ub3RpY2Ugc2VtYW50aWMgcGFzcyBkZWNpZGVzIG1lYW5pbmcsIG5vdCBzZW50ZW5jZS1jb3VudCBwYXJpdHkuCiAgICBmb3IgaSBpbiBzb3J0ZWQoc2V0KHJhbmdlKGxlbih6cykpKS1zZXQoaWkpKToKICAgICAgICBrPW1heChyYW5nZShsZW4oZ3JvdXBzKSksa2V5PWxhbWJkYSBrOm1heChzaW1baSxqXSBmb3IgaiBpbiBncm91cHNba11bMV0pKQogICAgICAgIGdyb3Vwc1trXVswXS5hcHBlbmQoaSkKICAgIGZvciBqIGluIHNvcnRlZChzZXQocmFuZ2UobGVuKGVzKSkpLXNldChqaikpOgogICAgICAgIGs9bWF4KHJhbmdlKGxlbihncm91cHMpKSxrZXk9bGFtYmRhIGs6bWF4KHNpbVtpLGpdIGZvciBpIGluIGdyb3Vwc1trXVswXSkpCiAgICAgICAgZ3JvdXBzW2tdWzFdLmFwcGVuZChqKQogICAgcmV0dXJuIFsoJyAnLmpvaW4oenNbaV0gZm9yIGkgaW4gc29ydGVkKGl6KSksJyAnLmpvaW4oZXNbal0gZm9yIGogaW4gc29ydGVkKGllKSksc2NvcmUpIGZvciBpeixpZSxzY29yZSBpbiBncm91cHNdCgoKZGVmIGRpZ2VzdCh0ZXh0KTpyZXR1cm4gaGFzaGxpYi5zaGEyNTYodGV4dC5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpjbGFzcyBDaGVja2VyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsY2FjaGU9Tm9uZSxjb25maWc9Tm9uZSxkZXZpY2U9Tm9uZSxwcm9ncmVzcz1wcmludCk6CiAgICAgICAgc2VsZi5wcm9ncmVzcz1wcm9ncmVzcwogICAgICAgIHNlbGYuZGV2aWNlPWRldmljZSBvciAoJ2N1ZGEnIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JykKICAgICAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCkKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZCgxMCkKICAgICAgICBzZWxmLmNvbmZpZz1jb25maWcgb3IganNvbi5sb2FkcyhQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoJ2NvbmZpZy5qc29uJykucmVhZF90ZXh0KCkpCiAgICAgICAgc2VsZi5jYWNoZV9wYXRoPVBhdGgoY2FjaGUpIGlmIGNhY2hlIGVsc2UgTm9uZQogICAgICAgIHNlbGYuY2FjaGU9anNvbi5sb2FkcyhzZWxmLmNhY2hlX3BhdGgucmVhZF90ZXh0KCkpIGlmIHNlbGYuY2FjaGVfcGF0aCBhbmQgc2VsZi5jYWNoZV9wYXRoLmV4aXN0cygpIGVsc2Uge30KICAgICAgICBzZWxmLnByb2dyZXNzKCdMb2FkaW5nIFYxMCBtb2RlbHMgKG5vIFY3IGZhbGxiYWNrKS4nLGZsdXNoPVRydWUpCiAgICAgICAgc2VsZi5sYWJzZT1TZW50ZW5jZVRyYW5zZm9ybWVyKE1PREVMX0lEU1snZW1iZWRkaW5nJ10scmV2aXNpb249TU9ERUxfUkVWSVNJT05TWydlbWJlZGRpbmcnXSxkZXZpY2U9c2VsZi5kZXZpY2UpCiAgICAgICAgc2VsZi50b2s9QXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoTU9ERUxfSURTWydubGknXSxyZXZpc2lvbj1NT0RFTF9SRVZJU0lPTlNbJ25saSddKQogICAgICAgIHNlbGYubmxpPUF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lEU1snbmxpJ10scmV2aXNpb249TU9ERUxfUkVWSVNJT05TWydubGknXSkudG8oc2VsZi5kZXZpY2UpLmV2YWwoKQogICAgICAgIHNlbGYubXRfdG9rPUF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lEU1sndHJhbnNsYXRpb24nXSxyZXZpc2lvbj1NT0RFTF9SRVZJU0lPTlNbJ3RyYW5zbGF0aW9uJ10sc3JjX2xhbmc9J3pob19IYW5zJykKICAgICAgICBzZWxmLm10PUF1dG9Nb2RlbEZvclNlcTJTZXFMTS5mcm9tX3ByZXRyYWluZWQoTU9ERUxfSURTWyd0cmFuc2xhdGlvbiddLHJldmlzaW9uPU1PREVMX1JFVklTSU9OU1sndHJhbnNsYXRpb24nXSkudG8oc2VsZi5kZXZpY2UpLmV2YWwoKQogICAgICAgIGxhYmVscz1zZWxmLm5saS5jb25maWcuaWQybGFiZWwKICAgICAgICBzZWxmLmxhYmVscz17bmFtZTppIGZvciBpLG5hbWUgaW4gbGFiZWxzLml0ZW1zKCl9CiAgICAgICAgYXNzZXJ0IGFsbCh4IGluIHNlbGYubGFiZWxzIGZvciB4IGluICgnZW50YWlsbWVudCcsJ25ldXRyYWwnLCdjb250cmFkaWN0aW9uJykpLGxhYmVscwogICAgICAgIHNlbGYucmV2aXNpb25zPU1PREVMX1JFVklTSU9OUy5jb3B5KCkKICAgICAgICBzZWxmLnByb2dyZXNzKCdWMTAgcmVhZHkgb24gJytzZWxmLmRldmljZSxmbHVzaD1UcnVlKQoKICAgIGRlZiBzYXZlX2NhY2hlKHNlbGYpOgogICAgICAgIGlmIHNlbGYuY2FjaGVfcGF0aDoKICAgICAgICAgICAgc2VsZi5jYWNoZV9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgdG1wPXNlbGYuY2FjaGVfcGF0aC53aXRoX3N1ZmZpeCgnLnRtcCcpCiAgICAgICAgICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2VsZi5jYWNoZSxlbnN1cmVfYXNjaWk9RmFsc2UpKQogICAgICAgICAgICB0bXAucmVwbGFjZShzZWxmLmNhY2hlX3BhdGgpCgogICAgZGVmIHRyYW5zbGF0ZShzZWxmLHRleHRzKToKICAgICAgICBtaXNzaW5nPWxpc3QoZGljdC5mcm9ta2V5cyh0IGZvciB0IGluIHRleHRzIGlmICdtdF9ubGxiX2hhbnNfdGVybXM6JytkaWdlc3QodCkgbm90IGluIHNlbGYuY2FjaGUpKQogICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLGxlbihtaXNzaW5nKSw4KToKICAgICAgICAgICAgYmF0Y2g9bWlzc2luZ1tzdGFydDpzdGFydCs4XQogICAgICAgICAgICBpbnA9c2VsZi5tdF90b2soW3RyYW5zbGF0aW9uX3NvdXJjZSh0KSBmb3IgdCBpbiBiYXRjaF0scmV0dXJuX3RlbnNvcnM9J3B0JyxwYWRkaW5nPVRydWUsdHJ1bmNhdGlvbj1GYWxzZSkudG8oc2VsZi5kZXZpY2UpCiAgICAgICAgICAgIGlmIGlucFsnaW5wdXRfaWRzJ10uc2hhcGVbMV0+NTEyOnJhaXNlIFZhbHVlRXJyb3IoJ1RyYW5zbGF0aW9uIHNlZ21lbnQgZXhjZWVkcyB0b2tlbiBsaW1pdDsgbm8gc2lsZW50IHRydW5jYXRpb24uJykKICAgICAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgICAgICAgICAgb3V0PXNlbGYubXQuZ2VuZXJhdGUoKippbnAsbWF4X25ld190b2tlbnM9MzAwLG51bV9iZWFtcz00LGZvcmNlZF9ib3NfdG9rZW5faWQ9c2VsZi5tdF90b2suY29udmVydF90b2tlbnNfdG9faWRzKCJlbmdfTGF0biIpKQogICAgICAgICAgICBmb3IgdCxpZHMscyBpbiB6aXAoYmF0Y2gsb3V0LHNlbGYubXRfdG9rLmJhdGNoX2RlY29kZShvdXQsc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKSk6CiAgICAgICAgICAgICAgICBpZiBsZW4oaWRzKT49MzAwIGFuZCBzZWxmLm10LmNvbmZpZy5lb3NfdG9rZW5faWQgbm90IGluIGlkcy50b2xpc3QoKVsxOl06CiAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZVsnbXRfbmxsYl9oYW5zX3Rlcm1zOicrZGlnZXN0KHQpXT17J2Vycm9yJzonVHJhbnNsYXRpb24gb3V0cHV0IGxpbWl0IHJlYWNoZWQ7IHRoZSBwYXNzYWdlIG5lZWRzIG1hbnVhbCByZXZpZXcuJ30KICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZVsnbXRfbmxsYl9oYW5zX3Rlcm1zOicrZGlnZXN0KHQpXT1zCiAgICAgICAgICAgIHNlbGYuc2F2ZV9jYWNoZSgpCiAgICAgICAgICAgIGlmIHN0YXJ0JTQwPT0wOnNlbGYucHJvZ3Jlc3MoZidUcmFuc2xhdGlvbiBjcm9zcy1jaGVjazoge21pbihzdGFydCs4LGxlbihtaXNzaW5nKSl9L3tsZW4obWlzc2luZyl9IG5ldyBwYXNzYWdlcycsZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4ge3Q6c2VsZi5jYWNoZVsnbXRfbmxsYl9oYW5zX3Rlcm1zOicrZGlnZXN0KHQpXSBmb3IgdCBpbiB0ZXh0c30KCiAgICBkZWYgbmxpX3Njb3JlcyhzZWxmLHBhaXJzKToKICAgICAgICBkZWYga2V5KHApOnJldHVybiAnbmxpOicrZGlnZXN0KGpzb24uZHVtcHMocCxlbnN1cmVfYXNjaWk9RmFsc2UpKQogICAgICAgIG1pc3Npbmc9bGlzdChkaWN0LmZyb21rZXlzKHAgZm9yIHAgaW4gcGFpcnMgaWYga2V5KHApIG5vdCBpbiBzZWxmLmNhY2hlKSkKICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCxsZW4obWlzc2luZyksMTIpOgogICAgICAgICAgICBiYXRjaD1taXNzaW5nW3N0YXJ0OnN0YXJ0KzEyXQogICAgICAgICAgICBpbnA9c2VsZi50b2soW3BbMF0gZm9yIHAgaW4gYmF0Y2hdLFtwWzFdIGZvciBwIGluIGJhdGNoXSxyZXR1cm5fdGVuc29ycz0ncHQnLHBhZGRpbmc9VHJ1ZSx0cnVuY2F0aW9uPUZhbHNlKS50byhzZWxmLmRldmljZSkKICAgICAgICAgICAgaWYgaW5wWydpbnB1dF9pZHMnXS5zaGFwZVsxXT41MTI6cmFpc2UgVmFsdWVFcnJvcignSW5mZXJlbmNlIHNlZ21lbnQgZXhjZWVkcyB0b2tlbiBsaW1pdDsgbm8gc2lsZW50IHRydW5jYXRpb24uJykKICAgICAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOnNjb3Jlcz1zZWxmLm5saSgqKmlucCkubG9naXRzLnNvZnRtYXgoLTEpLmNwdSgpLnRvbGlzdCgpCiAgICAgICAgICAgIGZvciBwLHMgaW4gemlwKGJhdGNoLHNjb3Jlcyk6c2VsZi5jYWNoZVtrZXkocCldPXtrOnNbaV0gZm9yIGssaSBpbiBzZWxmLmxhYmVscy5pdGVtcygpfQogICAgICAgICAgICBzZWxmLnNhdmVfY2FjaGUoKQogICAgICAgICAgICBpZiBzdGFydCUxMjA9PTA6c2VsZi5wcm9ncmVzcyhmJ0NvbnRleHR1YWwgaW5mZXJlbmNlOiB7bWluKHN0YXJ0KzEyLGxlbihtaXNzaW5nKSl9L3tsZW4obWlzc2luZyl9IG5ldyBwYWlycycsZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4ge3A6c2VsZi5jYWNoZVtrZXkocCldIGZvciBwIGluIHBhaXJzfQoKICAgIGRlZiBwcmVkaWN0X21hbnkoc2VsZixpbnB1dHMpOgogICAgICAgICMgT25seSB0aGVzZSBmaWVsZHMgY3Jvc3MgdGhlIHByZWRpY3RvciBib3VuZGFyeS4gUmVmZXJlbmNlcyBhcmUgbm90IHVzZWQuCiAgICAgICAgY2FzZXM9W3trOnJba10gZm9yIGsgaW4gKCdjYXNlX2lkJywnY2hpbmVzZScsJ2VuZ2xpc2gnKX0gZm9yIHIgaW4gaW5wdXRzXQogICAgICAgIHdvcms9W107IGludmFsaWQ9e30KICAgICAgICBmb3IgaSxyIGluIGVudW1lcmF0ZShjYXNlcyk6CiAgICAgICAgICAgIHosZT1ub3JtYWwoclsnY2hpbmVzZSddKSxub3JtYWwoclsnZW5nbGlzaCddKQogICAgICAgICAgICBpZiBub3QgeiBvciBub3QgZToKICAgICAgICAgICAgICAgIGludmFsaWRbaV09J0JvdGggbGFuZ3VhZ2UgdmVyc2lvbnMgYXJlIHJlcXVpcmVkLic7Y29udGludWUKICAgICAgICAgICAgaWYgbWF4KGxlbih6KSxsZW4oZSkpPjYwMDA6CiAgICAgICAgICAgICAgICBpbnZhbGlkW2ldPSdUaGUgNiwwMDAtY2hhcmFjdGVyIGlucHV0IGxpbWl0IHdhcyBleGNlZWRlZC4nO2NvbnRpbnVlCiAgICAgICAgICAgICMgU2hvcnQgbm90aWNlcyBhcmUgY29tcGFyZWQgYXMgY29tcGxldGUgY29udGV4dHMuIExvbmcgbm90aWNlcyBhcmUKICAgICAgICAgICAgIyBleHBsaWNpdGx5IHVucmVzb2x2ZWQgcmF0aGVyIHRoYW4gc2lsZW50bHkgZHJvcHBpbmcgbGF0ZXIgdGV4dC4KICAgICAgICAgICAgaWYgbGVuKHNlbGYubXRfdG9rKHRyYW5zbGF0aW9uX3NvdXJjZSh6KSlbJ2lucHV0X2lkcyddKT40ODAgb3IgbGVuKHNlbGYudG9rKHosZSlbJ2lucHV0X2lkcyddKT40ODA6CiAgICAgICAgICAgICAgICBpbnZhbGlkW2ldPSdUaGlzIG5vdGljZSBleGNlZWRzIFYxMOKAmXMgdmFsaWRhdGVkIGNvbnRleHQgd2luZG93OyBzcGxpdCBpdCBpbnRvIHNob3J0ZXIgY29tcGxldGUgc2VjdGlvbnMgZm9yIHJldmlldy4nO2NvbnRpbnVlCiAgICAgICAgICAgIHdvcmsuYXBwZW5kKChpLHosZSkpCiAgICAgICAgdGV4dHM9bGlzdChkaWN0LmZyb21rZXlzKFt0IGZvciBfLHosZSBpbiB3b3JrIGZvciB0IGluIFt6LGUsKnNlbnRlbmNlcyh6KSwqc2VudGVuY2VzKGUpXV0pKQogICAgICAgIHZlY3RvcnM9c2VsZi5sYWJzZS5lbmNvZGUodGV4dHMsbm9ybWFsaXplX2VtYmVkZGluZ3M9VHJ1ZSxzaG93X3Byb2dyZXNzX2Jhcj1GYWxzZSxiYXRjaF9zaXplPTE2KSBpZiB0ZXh0cyBlbHNlIFtdCiAgICAgICAgZW1iZWRkaW5ncz1kaWN0KHppcCh0ZXh0cyx2ZWN0b3JzKSkKICAgICAgICBzZW50ZW5jZV90cmFuc2xhdGlvbnM9c2VsZi50cmFuc2xhdGUobGlzdChkaWN0LmZyb21rZXlzKHQgZm9yIF8seixfIGluIHdvcmsgZm9yIHQgaW4gc2VudGVuY2VzKHopKSkpCiAgICAgICAgdmFsaWQ9W10KICAgICAgICBmb3IgaSx6LGUgaW4gd29yazoKICAgICAgICAgICAgaWYgYW55KG5vdCBpc2luc3RhbmNlKHNlbnRlbmNlX3RyYW5zbGF0aW9uc1t0XSxzdHIpIGZvciB0IGluIHNlbnRlbmNlcyh6KSk6CiAgICAgICAgICAgICAgICBpbnZhbGlkW2ldPSdUaGUgdHJhbnNsYXRpb24gY3Jvc3MtY2hlY2sgY291bGQgbm90IGNvdmVyIHRoaXMgbm90aWNlOyBtYW51YWwgcmV2aWV3IGlzIG5lZWRlZC4nCiAgICAgICAgICAgIGVsc2U6dmFsaWQuYXBwZW5kKChpLHosZSkpCiAgICAgICAgd29yaz12YWxpZAogICAgICAgIHRyYW5zbGF0aW9ucz17ejonICcuam9pbihzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0gZm9yIHQgaW4gc2VudGVuY2VzKHopKSBmb3IgXyx6LF8gaW4gd29ya30KICAgICAgICB2YWxpZD1bXQogICAgICAgIGZvciBpLHosZSBpbiB3b3JrOgogICAgICAgICAgICBpZiBsZW4oc2VsZi50b2sodHJhbnNsYXRpb25zW3pdLGUpWydpbnB1dF9pZHMnXSk+NTAwOgogICAgICAgICAgICAgICAgaW52YWxpZFtpXT0nVGhlIHRyYW5zbGF0ZWQgY29tcGFyaXNvbiBleGNlZWRzIHRoZSBtb2RlbCBjb250ZXh0OyBtYW51YWwgcmV2aWV3IGlzIG5lZWRlZC4nCiAgICAgICAgICAgIGVsc2U6dmFsaWQuYXBwZW5kKChpLHosZSkpCiAgICAgICAgd29yaz12YWxpZAogICAgICAgIGFsaWduZWQ9e2k6YWxpZ25lZF9ncm91cHMoc2VudGVuY2VzKHopLHNlbnRlbmNlcyhlKSxlbWJlZGRpbmdzKSBmb3IgaSx6LGUgaW4gd29ya30KICAgICAgICBxdWFsaXR5X3RleHRzPWxpc3QoZGljdC5mcm9ta2V5cyhbdHJhbnNsYXRpb25zW3pdIGZvciBfLHosXyBpbiB3b3JrXStbdiBmb3IgdiBpbiBzZW50ZW5jZV90cmFuc2xhdGlvbnMudmFsdWVzKCkgaWYgaXNpbnN0YW5jZSh2LHN0cildK1thIGZvciBpLHosZSBpbiB3b3JrIGZvciBhLGIsc2ltIGluIGFsaWduZWRbaV1dK1snICcuam9pbihzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0gZm9yIHQgaW4gc2VudGVuY2VzKHopIGlmIHQgaW4gYSkgZm9yIGkseixlIGluIHdvcmsgZm9yIGEsYixzaW0gaW4gYWxpZ25lZFtpXV0pKQogICAgICAgIHF1YWxpdHlfdmVjdG9ycz1zZWxmLmxhYnNlLmVuY29kZShxdWFsaXR5X3RleHRzLG5vcm1hbGl6ZV9lbWJlZGRpbmdzPVRydWUsc2hvd19wcm9ncmVzc19iYXI9RmFsc2UsYmF0Y2hfc2l6ZT0xNikgaWYgcXVhbGl0eV90ZXh0cyBlbHNlIFtdCiAgICAgICAgZW1iZWRkaW5ncy51cGRhdGUoZGljdCh6aXAocXVhbGl0eV90ZXh0cyxxdWFsaXR5X3ZlY3RvcnMpKSkKICAgICAgICBwYWlycz1bXQogICAgICAgIGZvciBfLHosZSBpbiB3b3JrOgogICAgICAgICAgICBtdD10cmFuc2xhdGlvbnNbel0KICAgICAgICAgICAgcGFpcnMuZXh0ZW5kKFsoeixlKSwoZSx6KSwobXQsZSksKGUsbXQpXSkKICAgICAgICAgICAgZm9yIGEsYixzaW0gaW4gYWxpZ25lZFtfXToKICAgICAgICAgICAgICAgIGFtPScgJy5qb2luKHNlbnRlbmNlX3RyYW5zbGF0aW9uc1t0XSBmb3IgdCBpbiBzZW50ZW5jZXMoeikgaWYgdCBpbiBhKQogICAgICAgICAgICAgICAgcGFpcnMuZXh0ZW5kKFsoYSxiKSwoYixhKSwoYW0sYiksKGIsYW0pXSkKICAgICAgICBzY29yZXM9c2VsZi5ubGlfc2NvcmVzKHBhaXJzKQogICAgICAgIG91dHB1dD17fQogICAgICAgIGZvciBpLHosZSBpbiB3b3JrOgogICAgICAgICAgICBtdD10cmFuc2xhdGlvbnNbel0KICAgICAgICAgICAgcD1bc2NvcmVzW3hdIGZvciB4IGluIFsoeixlKSwoZSx6KSwobXQsZSksKGUsbXQpXV0KICAgICAgICAgICAgc2lnbmFsPWRpY3QoY3Jvc3M9W3hbJ2NvbnRyYWRpY3Rpb24nXSBmb3IgeCBpbiBwWzoyXV0sY3Jvc3NfZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBbOjJdXSxjcm9zc19uZXV0cmFsPVt4WyduZXV0cmFsJ10gZm9yIHggaW4gcFs6Ml1dLAogICAgICAgICAgICAgICAgICAgICAgICB0cmFuc2xhdGVkPVt4Wydjb250cmFkaWN0aW9uJ10gZm9yIHggaW4gcFsyOl1dLAogICAgICAgICAgICAgICAgICAgICAgICBlbnRhaWxtZW50PVt4WydlbnRhaWxtZW50J10gZm9yIHggaW4gcFsyOl1dLAogICAgICAgICAgICAgICAgICAgICAgICBuZXV0cmFsPVt4WyduZXV0cmFsJ10gZm9yIHggaW4gcFsyOl1dLAogICAgICAgICAgICAgICAgICAgICAgICBzaW1pbGFyaXR5PWZsb2F0KGVtYmVkZGluZ3Nbel1AZW1iZWRkaW5nc1tlXSksdHJhbnNsYXRpb25fcXVhbGl0eT1mbG9hdChlbWJlZGRpbmdzW3pdQGVtYmVkZGluZ3NbbXRdKSx0cmFuc2xhdGlvbj1tdCkKICAgICAgICAgICAgZmFjdHM9ZGlyZWN0X2V2aWRlbmNlKHosZSkKICAgICAgICAgICAgZ3JvdXBzPWFsaWduZWRbaV0KICAgICAgICAgICAgIyBQYXNzYWdlLWxldmVsIGZhY3RzIGRldGVjdCBzd2FwcGVkIHJvbGVzIGV2ZW4gd2hlbiBkb2N1bWVudC1sZXZlbAogICAgICAgICAgICAjIHZhbHVlIHNldHMgYWdyZWUuIFdlYWsgYWxpZ25tZW50cyBkbyBub3QgZXN0YWJsaXNoIGEgZmFjdHVhbCBlcnJvci4KICAgICAgICAgICAgZm9yIGEsYixzaW0gaW4gZ3JvdXBzOgogICAgICAgICAgICAgICAgaWYgc2ltPj1zZWxmLmNvbmZpZ1snZmFjdF9hbGlnbm1lbnRfbWluJ106CiAgICAgICAgICAgICAgICAgICAgZmFjdHMuZXh0ZW5kKGRpcmVjdF9ldmlkZW5jZShhLGIpKQogICAgICAgICAgICBmYWN0cz1saXN0KHtqc29uLmR1bXBzKHgsc29ydF9rZXlzPVRydWUsZW5zdXJlX2FzY2lpPUZhbHNlKTp4IGZvciB4IGluIGZhY3RzfS52YWx1ZXMoKSkKICAgICAgICAgICAgcmVhc29ucz1bXQogICAgICAgICAgICBmb3IgZiBpbiBmYWN0czoKICAgICAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKGRpY3Qoa2luZD0nZXhwbGljaXRfZmFjdCcsY2F0ZWdvcnk9ZlsnY2F0ZWdvcnknXSwKICAgICAgICAgICAgICAgICAgICB3aHk9ZiJ7ZlsnY2F0ZWdvcnknXX0gZGlmZmVyczogRW5nbGlzaCB7ZlsnZW5nbGlzaF92YWx1ZSddfSBbe2ZbJ2VuZ2xpc2hfc3BhbiddfV07IENoaW5lc2Uge2ZbJ2NoaW5lc2VfdmFsdWUnXX0gW3tmWydjaGluZXNlX3NwYW4nXX1dLiIsCiAgICAgICAgICAgICAgICAgICAgKip7azp2IGZvciBrLHYgaW4gZi5pdGVtcygpIGlmIGshPSdjYXRlZ29yeSd9KSkKICAgICAgICAgICAgc2VtYW50aWM9c2VsZi5zZW1hbnRpY19kZWNpc2lvbihzaWduYWwpCiAgICAgICAgICAgIHBhc3NhZ2Vfc2lnbmFscz1bXQogICAgICAgICAgICBmb3IgYSxiLHNpbSBpbiBncm91cHM6CiAgICAgICAgICAgICAgICBhbT0nICcuam9pbihzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0gZm9yIHQgaW4gc2VudGVuY2VzKHopIGlmIHQgaW4gYSkKICAgICAgICAgICAgICAgIHBzPVtzY29yZXNbeF0gZm9yIHggaW4gWyhhLGIpLChiLGEpLChhbSxiKSwoYixhbSldXQogICAgICAgICAgICAgICAgc3M9ZGljdChjcm9zcz1beFsnY29udHJhZGljdGlvbiddIGZvciB4IGluIHBzWzoyXV0sY3Jvc3NfZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBzWzoyXV0sY3Jvc3NfbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBzWzoyXV0sZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBzWzI6XV0sbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBzWzI6XV0sdHJhbnNsYXRlZD1beFsnY29udHJhZGljdGlvbiddIGZvciB4IGluIHBzWzI6XV0sc2ltaWxhcml0eT1zaW0sdHJhbnNsYXRpb25fcXVhbGl0eT1mbG9hdChlbWJlZGRpbmdzW2FdQGVtYmVkZGluZ3NbYW1dKSxlbmdsaXNoX3NwYW49YixjaGluZXNlX3NwYW49YSx0cmFuc2xhdGlvbj1hbSkKICAgICAgICAgICAgICAgIHBhc3NhZ2Vfc2lnbmFscy5hcHBlbmQoc3MpCiAgICAgICAgICAgIHNpZ25hbFsncGFzc2FnZXMnXT1wYXNzYWdlX3NpZ25hbHMKICAgICAgICAgICAgZm9jdXM9Tm9uZQogICAgICAgICAgICBzdXBwb3J0ZWQ9W3NzIGZvciBzcyBpbiBwYXNzYWdlX3NpZ25hbHMgaWYgc3NbJ3NpbWlsYXJpdHknXT49c2VsZi5jb25maWdbJ3NlbWFudGljX2FsaWdubWVudF9taW4nXSBhbmQgc2VsZi5zZW1hbnRpY19kZWNpc2lvbihzcyldCiAgICAgICAgICAgIGlmIHN1cHBvcnRlZDoKICAgICAgICAgICAgICAgIGZvY3VzPW1heChzdXBwb3J0ZWQsa2V5PWxhbWJkYSBzczptaW4oc3NbJ3RyYW5zbGF0ZWQnXSkrbWluKHNzWydjcm9zcyddKSkKICAgICAgICAgICAgICAgIHNlbWFudGljPXNlbGYuc2VtYW50aWNfZGVjaXNpb24oZm9jdXMpCiAgICAgICAgICAgIGVsaWYgbGVuKGdyb3Vwcyk+MSBhbmQgbm90IChtaW4oc2lnbmFsWydjcm9zcyddKT49c2VsZi5jb25maWdbJ3dob2xlX2Nyb3NzX21pbiddIGFuZCBtaW4oc2lnbmFsWyd0cmFuc2xhdGVkJ10pPj1zZWxmLmNvbmZpZ1snd2hvbGVfbXRfbWluJ10pOgogICAgICAgICAgICAgICAgIyBCb3RoIG1vZGVscyBtdXN0IGNvcnJvYm9yYXRlIHRoZSBTQU1FIHBhc3NhZ2UuIFR3byB3ZWFrCiAgICAgICAgICAgICAgICAjIGRvY3VtZW50IHNjb3JlcyBjYW4gcmVmZXIgdG8gdW5yZWxhdGVkIHNlbnRlbmNlcy4KICAgICAgICAgICAgICAgIHNlbWFudGljPU5vbmUKICAgICAgICAgICAgaWYgc2VtYW50aWM6CiAgICAgICAgICAgICAgICAjIERpZmYgaXMgYSBuYXZpZ2F0aW9uIGFpZCwgbm90IGFuIGFzc2VydGVkIHRyYW5zbGF0aW9uIHZlcmRpY3QuCiAgICAgICAgICAgICAgICBlZGl0cz1bXQogICAgICAgICAgICAgICAgZmU9Zm9jdXNbJ2VuZ2xpc2hfc3BhbiddIGlmIGZvY3VzIGVsc2UgZQogICAgICAgICAgICAgICAgZno9Zm9jdXNbJ2NoaW5lc2Vfc3BhbiddIGlmIGZvY3VzIGVsc2UgegogICAgICAgICAgICAgICAgZm10PWZvY3VzWyd0cmFuc2xhdGlvbiddIGlmIGZvY3VzIGVsc2UgbXQKICAgICAgICAgICAgICAgIGE9ZmUuc3BsaXQoKTtiPWZtdC5zcGxpdCgpCiAgICAgICAgICAgICAgICBmb3IgdGFnLGwsaCxqLGsgaW4gZGlmZmxpYi5TZXF1ZW5jZU1hdGNoZXIoYT1hLGI9YixhdXRvanVuaz1GYWxzZSkuZ2V0X29wY29kZXMoKToKICAgICAgICAgICAgICAgICAgICBpZiB0YWchPSdlcXVhbCc6ZWRpdHMuYXBwZW5kKHsnZW5nbGlzaCc6JyAnLmpvaW4oYVtsOmhdKSwnbWFjaGluZV9iYWNrdHJhbnNsYXRpb24nOicgJy5qb2luKGJbajprXSl9KQogICAgICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZGljdChraW5kPSdjb250ZXh0dWFsX21vZGVsJyxjYXRlZ29yeT1zZW1hbnRpYyx3aHk9Zid7c2VtYW50aWN9LiBSZXZpZXcgdGhlIHF1b3RlZCBwYXNzYWdlczsgdGhlIG1hY2hpbmUgYmFjay10cmFuc2xhdGlvbiBpcyBzdXBwb3J0aW5nIGV2aWRlbmNlLCBub3QgYSByZWZlcmVuY2UgdHJhbnNsYXRpb24uJyxlbmdsaXNoX3NwYW49ZmUsY2hpbmVzZV9zcGFuPWZ6LG1hY2hpbmVfYmFja3RyYW5zbGF0aW9uPWZtdCxjb21wYXJpc29uX2VkaXRzPWVkaXRzKSkKICAgICAgICAgICAgb3V0cHV0W2ldPWRpY3QoY2FzZV9pZD1jYXNlc1tpXVsnY2FzZV9pZCddLGZsYWdnZWQ9Ym9vbChyZWFzb25zKSx3YXJuaW5ncz1yZWFzb25zLHNpZ25hbHM9c2lnbmFsLAogICAgICAgICAgICAgICAgZGVjaXNpb25fcGF0aD0ndjEwX2NvbnRleHR1YWxfY29tcGFyaXNvbicsY292ZXJhZ2U9J2NvbXBsZXRlX25vdGljZScsdjdfZmFsbGJhY2s9RmFsc2UpCiAgICAgICAgZm9yIGksbWVzc2FnZSBpbiBpbnZhbGlkLml0ZW1zKCk6CiAgICAgICAgICAgIG91dHB1dFtpXT1kaWN0KGNhc2VfaWQ9Y2FzZXNbaV1bJ2Nhc2VfaWQnXSxmbGFnZ2VkPVRydWUsd2FybmluZ3M9W2RpY3Qoa2luZD0ndW5yZXNvbHZlZCcsY2F0ZWdvcnk9J2lucHV0IGNvdmVyYWdlJyx3aHk9bWVzc2FnZSxlbmdsaXNoX3NwYW49Y2FzZXNbaV1bJ2VuZ2xpc2gnXSxjaGluZXNlX3NwYW49Y2FzZXNbaV1bJ2NoaW5lc2UnXSldLGRlY2lzaW9uX3BhdGg9J3YxMF91bnJlc29sdmVkJyxjb3ZlcmFnZT0ndW5yZXNvbHZlZCcsdjdfZmFsbGJhY2s9RmFsc2UpCiAgICAgICAgcmV0dXJuIFtvdXRwdXRbaV0gZm9yIGkgaW4gcmFuZ2UobGVuKGNhc2VzKSldCgogICAgZGVmIHNlbWFudGljX2RlY2lzaW9uKHNlbGYscyk6CiAgICAgICAgYz1zZWxmLmNvbmZpZwogICAgICAgIGlmIHMuZ2V0KCd0cmFuc2xhdGlvbl9xdWFsaXR5JywxKTxjWyd0cmFuc2xhdGlvbl9xdWFsaXR5X21pbiddOnJldHVybiBOb25lCiAgICAgICAgeD1tYXgoc1snY3Jvc3MnXSk7bT1tYXgoc1sndHJhbnNsYXRlZCddKTtiPW1pbihzWyd0cmFuc2xhdGVkJ10pCiAgICAgICAgaWYgYj49Y1snbXRfYmlkaXJlY3Rpb25hbCddIGFuZCB4Pj1jWydjcm9zc19zdXBwb3J0J106CiAgICAgICAgICAgIHJldHVybiAnVGhlIGNvbXBsZXRlIHZlcnNpb25zIGdpdmUgY29uZmxpY3RpbmcgaW5mb3JtYXRpb24nCiAgICAgICAgaWYgbT49Y1snbXRfb25ld2F5J10gYW5kIHg+PWNbJ2Nyb3NzX3N1cHBvcnQnXToKICAgICAgICAgICAgcmV0dXJuICdUd28gc2VtYW50aWMgY29tcGFyaXNvbnMgaW5kaWNhdGUgYSBtZWFuaW5nIGRpZmZlcmVuY2UnCiAgICAgICAgaWYgc1snc2ltaWxhcml0eSddPGNbJ3NpbWlsYXJpdHlfbWF4J10gYW5kIG0+PWNbJ210X2xvd19zaW1pbGFyaXR5J10gYW5kIHg+PWNbJ2Nyb3NzX3N1cHBvcnQnXToKICAgICAgICAgICAgcmV0dXJuICdUaGUgdmVyc2lvbnMgbWF5IGRlc2NyaWJlIGRpZmZlcmVudCBmYWN0cyBvciBpbnN0cnVjdGlvbnMnCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgY2hlY2tfbm90aWNlKHNlbGYsY2hpbmVzZV90ZXh0LGVuZ2xpc2hfdGV4dCk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgICAgIHI9c2VsZi5wcmVkaWN0X21hbnkoW2RpY3QoY2FzZV9pZD0naW50ZXJhY3RpdmUnLGNoaW5lc2U9Y2hpbmVzZV90ZXh0LGVuZ2xpc2g9ZW5nbGlzaF90ZXh0KV0pWzBdCiAgICAgICAgd2FybmluZ3M9clsnd2FybmluZ3MnXSBvciBbZGljdCh3aHk9JycsY2F0ZWdvcnk9JycpXQogICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUoW2RpY3Qoc3RhdHVzPSfimqDvuI8gQ2hlY2snIGlmIHJbJ2ZsYWdnZWQnXSBlbHNlICfinIUgTm8gZGlzY3JlcGFuY3kgZm91bmQnLGNoaW5lc2U9Y2hpbmVzZV90ZXh0LGVuZ2xpc2g9ZW5nbGlzaF90ZXh0LGRlY2lzaW9uX3BhdGg9clsnZGVjaXNpb25fcGF0aCddLGNvdmVyYWdlPXJbJ2NvdmVyYWdlJ10sKip3KSBmb3IgdyBpbiB3YXJuaW5nc10pCg==', 'facts.py': 'IiIiU21hbGwsIGNvbnRleHQtaW5kZXBlbmRlbnQgYmlsaW5ndWFsIGZhY3RzLiBOZXZlciBjb21wYXJlIGJhcmUgbmVnYXRpb24gY291bnRzLgoKT25seSBhIHVuaXF1ZWx5IHJlc29sdmVkIHByb3BlcnR5IGluIEJPVEggcGFzc2FnZXMgY2FuIGVzdGFibGlzaCBhIGNvbmZsaWN0LgpUaGVzZSBhcmUgY2hlY2tzLCBub3QgYSBjbGFpbSBvZiBleGhhdXN0aXZlIGxpbmd1aXN0aWMgY292ZXJhZ2UuCiIiIgppbXBvcnQgcmUKaW1wb3J0IHVuaWNvZGVkYXRhCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gZGVjaW1hbCBpbXBvcnQgRGVjaW1hbAoKCmRlZiBub3JtYWwodGV4dCk6CiAgICByZXR1cm4gdW5pY29kZWRhdGEubm9ybWFsaXplKCdORktDJywgdGV4dCkucmVwbGFjZSgn4oCZJywgIiciKS5zdHJpcCgpCgpDTiA9IGRpY3QoemlwKCfpm7bjgIfil4vkuIDkuozlhankuInlm5vkupTlha3kuIPlhavkuZ0nLCBbMCwwLDAsMSwyLDIsMyw0LDUsNiw3LDgsOV0pKQpOVSA9IHsn5Y2BJzoxMCwgJ+eZvic6MTAwLCAn5Y2DJzoxMDAwLCAn6JCsJzoxMDAwMH0KTlVNID0gcicoPzpcZCsoPzpbLC5dXGQrKSp8W+mbtuOAh+KXi+S4gOS6jOWFqeS4ieWbm+S6lOWFreS4g+WFq+S5neWNgeeZvuWNg+iQrF0rKScKRU5OVU0gPSBkaWN0KHppcCgnemVybyBvbmUgdHdvIHRocmVlIGZvdXIgZml2ZSBzaXggc2V2ZW4gZWlnaHQgbmluZSB0ZW4gZWxldmVuIHR3ZWx2ZSB0aGlydGVlbiBmb3VydGVlbiBmaWZ0ZWVuIHNpeHRlZW4gc2V2ZW50ZWVuIGVpZ2h0ZWVuIG5pbmV0ZWVuIHR3ZW50eScuc3BsaXQoKSxyYW5nZSgyMSkpKQoKZGVmIG51bWJlcihzKToKICAgIHM9cy5sb3dlcigpLnJlcGxhY2UoJywnLCcnKQogICAgaWYgcyBpbiBFTk5VTTpyZXR1cm4gRGVjaW1hbChFTk5VTVtzXSkKICAgIGlmIHJlLmZ1bGxtYXRjaChyJ1xkKyg/OlwuXGQrKT8nLHMpOnJldHVybiBEZWNpbWFsKHMpCiAgICBpZiBhbGwoeCBpbiBDTiBmb3IgeCBpbiBzKTpyZXR1cm4gRGVjaW1hbCgnJy5qb2luKHN0cihDTlt4XSkgZm9yIHggaW4gcykpCiAgICB0b3RhbD1jdXI9c2VjdGlvbj0wCiAgICBmb3IgeCBpbiBzOgogICAgICAgIGlmIHggaW4gQ046Y3VyPUNOW3hdCiAgICAgICAgZWxpZiB4IGluIE5VOgogICAgICAgICAgICBpZiBOVVt4XT09MTAwMDA6dG90YWwrPShzZWN0aW9uK2N1cikqMTAwMDA7c2VjdGlvbj1jdXI9MAogICAgICAgICAgICBlbHNlOnNlY3Rpb24rPShjdXIgb3IgMSkqTlVbeF07Y3VyPTAKICAgICAgICBlbHNlOnJhaXNlIFZhbHVlRXJyb3IocykKICAgIHJldHVybiBEZWNpbWFsKHRvdGFsK3NlY3Rpb24rY3VyKQoKIyBDbG9zZWQtdmFsdWUgY29uY2VwdHM7IHdvcmQtbGV2ZWwgYm91bmRhcmllcyBhdm9pZCBlLmcuICdyZWQnIGluc2lkZSAncmVxdWlyZWQnLgpDT05DRVBUUyA9IHsKICdjb2xvdXInOiBbKCdyZWQnLHIncmVkJyxyJ+e0heiJsicpLCgnYmx1ZScscidibHVlJyxyJ+iXjeiJsicpLCgnZ3JlZW4nLHInZ3JlZW4nLHIn57ag6ImyJyksKCd5ZWxsb3cnLHIneWVsbG93JyxyJ+m7g+iJsicpLCgnYmxhY2snLHInYmxhY2snLHIn6buR6ImyJyksKCd3aGl0ZScscid3aGl0ZScscifnmb3oibInKSwoJ29yYW5nZScscidvcmFuZ2UnLHIn5qmZ6ImyJyksKCdwdXJwbGUnLHIncHVycGxlJyxyJ+e0q+iJsicpXSwKICdsYW5ndWFnZSc6IFsoJ0VuZ2xpc2gnLHInRW5nbGlzaCcscifoi7Hmlod86Iux6KqeJyksKCdDaGluZXNlJyxyJ0NoaW5lc2UnLHIn5Lit5paHfOa8ouiqnicpLCgnQ2FudG9uZXNlJyxyJ0NhbnRvbmVzZScscifnsrXoqp585buj5p2x6KmxJyksKCdNYW5kYXJpbicscidNYW5kYXJpbnxQdXRvbmdodWEnLHIn5pmu6YCa6KmxfOWci+iqnicpLCgnSmFwYW5lc2UnLHInSmFwYW5lc2UnLHIn5pel5paHfOaXpeiqnicpLCgnS29yZWFuJyxyJ0tvcmVhbicscifpn5Pmlod86Z+T6KqeJyksKCdGcmVuY2gnLHInRnJlbmNoJyxyJ+azleaWh3zms5Xoqp4nKSwoJ1NwYW5pc2gnLHInU3BhbmlzaCcscifopb/nj63niZnmlod86KW/54+t54mZ6KqeJyldLAogJ29yaWVudGF0aW9uJzogWygncG9ydHJhaXQnLHIncG9ydHJhaXQnLHIn55u05ZCRfOe4seWQkScpLCgnbGFuZHNjYXBlJyxyJ2xhbmRzY2FwZScscifmqavlkJEnKV0sCiAnbWF0ZXJpYWwnOiBbKCdnbGFzcycscidnbGFzcycscifnjrvnkoMnKSwoJ3BsYXN0aWMnLHIncGxhc3RpYycscifloZHohqB85aGR5paZfOiGoOaovScpLCgnbWV0YWwnLHInbWV0YWwnLHIn6YeR5bGsJyksKCdhbHVtaW5pdW0nLHInYWx1bWluKD86aXVtfHVtKScscifpi4EnKSwoJ3dvb2QnLHInd29vZCg/OmVuKT8nLHIn5pyo6KO9fOacqOadkCcpLCgncGFwZXInLHIncGFwZXInLHIn57SZ6KO9fOe0meW8tScpXSwKICdmaWxlIGZvcm1hdCc6IFsoeCwgeCwgeCkgZm9yIHggaW4gWydQREYnLCdDU1YnLCdET0NYJywnWExTWCcsJ0pQRUcnLCdQTkcnLCdaSVAnXV0sCiAnd2Vla2RheSc6IFsoZSxlK3Incz8nLHInKD865pif5pyffOmAsXzlkagpJyt6KSBmb3IgZSx6IGluIHppcChbJ01vbmRheScsJ1R1ZXNkYXknLCdXZWRuZXNkYXknLCdUaHVyc2RheScsJ0ZyaWRheScsJ1NhdHVyZGF5JywnU3VuZGF5J10sWyfkuIAnLCfkuownLCfkuIknLCflm5snLCfkupQnLCflha0nLCdb5pel5aSpXSddKV0sCiAnZnJlcXVlbmN5JzogWygnZGFpbHknLHInZGFpbHl8ZXZlcnkgZGF5JyxyJ+avj+aXpXzmr4/lpKknKSwoJ3dlZWtseScscid3ZWVrbHl8ZXZlcnkgd2Vlaycscifmr4/pgLF85q+P5pif5pyffOavj+WRqCcpLCgnbW9udGhseScscidtb250aGx5fGV2ZXJ5IG1vbnRoJyxyJ+avj+aciCcpLCgncXVhcnRlcmx5JyxyJ3F1YXJ0ZXJseXxldmVyeSB0aHJlZSBtb250aHMnLHIn5q+P5a2jfOavj+S4ieWAi+aciCcpLCgneWVhcmx5JyxyJ3llYXJseXxhbm51YWxseXxhbm51YWx8ZXZlcnkgeWVhcicscifmr4/lubR85LiA5bm05LiA5bqmJyldLAogJ2RvY3VtZW50IGZvcm0nOiBbKCdvcmlnaW5hbCcscidvcmlnaW5hbCg/OnMpPycscifmraPmnKx85Y6f5Lu2JyksKCdjb3B5JyxyJ2NvcCg/Onl8aWVzKXxwaG90b2NvcCg/Onl8aWVzKScsciflia/mnKx85b2x5Y2w5pysJyldLAp9CgpkZWYgZXZpZGVuY2UoY2F0ZWdvcnksIHosIGUpOgogICAgcmV0dXJuIGRpY3QoY2F0ZWdvcnk9Y2F0ZWdvcnksY2hpbmVzZV92YWx1ZT1zdHIoelswXSksZW5nbGlzaF92YWx1ZT1zdHIoZVswXSksY2hpbmVzZV9zcGFuPXpbMV0sZW5nbGlzaF9zcGFuPWVbMV0pCgpkZWYgdW5pcXVlKGhpdHMpOgogICAgdmFsdWVzPXt4WzBdIGZvciB4IGluIGhpdHN9CiAgICByZXR1cm4gaGl0c1swXSBpZiBsZW4odmFsdWVzKT09MSBlbHNlIE5vbmUKCgpkZWYgY29uY2VwdF9mYWN0cyh0ZXh0LCBsYW5nKToKICAgIGZvdW5kPXt9CiAgICBmb3IgY2F0ZWdvcnksIGVudHJpZXMgaW4gQ09OQ0VQVFMuaXRlbXMoKToKICAgICAgICBoaXRzPVtdCiAgICAgICAgZm9yIHZhbHVlLGVwLHpwIGluIGVudHJpZXM6CiAgICAgICAgICAgIHBhdD1yJ1xiKD86JytlcCtyJylcYicgaWYgbGFuZz09J2VuJyBlbHNlIHpwCiAgICAgICAgICAgIGhpdHMuZXh0ZW5kKCh2YWx1ZSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXQsdGV4dCxyZS5JKSkKICAgICAgICBmPXVuaXF1ZShoaXRzKQogICAgICAgIGlmIGY6Zm91bmRbY2F0ZWdvcnldPWYKICAgIHJldHVybiBmb3VuZAoKIyBRdWFudGl0eSB1bml0cyBpbmNsdWRlIGEgZGltZW5zaW9uIGFuZCBjb252ZXJzaW9uIG11bHRpcGxpZXIuIENvcnJlY3QgY2hhbmdlcwojIHN1Y2ggYXMgMSBtIC8gMTAwIGNtIGFyZSBlcXVpdmFsZW50OyBwZXJjZW50YWdlcyBpbiDnmb7liIbmr5QgYXJlIG5vdCB0aGUgbnVtYmVyIDEwMC4KVU5JVFM9WwogKCdsZW5ndGgnLCcxJyxyJ20oPzpldHJlfGV0ZXIpcz98bScscifnsbN85YWs5bC6JyksCiAoJ2xlbmd0aCcsJy4wMScscidjZW50aW0oPzpldHJlfGV0ZXIpcz98Y20nLHIn5Y6Y57GzfOWFrOWIhicpLAogKCdsZW5ndGgnLCcuMDAxJyxyJ21pbGxpbSg/OmV0cmV8ZXRlcilzP3xtbScscifmr6vnsbMnKSwKICgnbGVuZ3RoJywnMTAwMCcscidraWxvbSg/OmV0cmV8ZXRlcilzP3xrbScsciflhazph4x85Y2D57GzJyksCiAoJ21hc3MnLCcxJyxyJ2tpbG9ncmFtcz98a2cnLHIn5YWs5pakfOWNg+WFiycpLCgnbWFzcycsJy4wMDEnLHInZ3JhbXM/fGcnLHIn5YWLfOWFrOWFiycpLAogKCdkdXJhdGlvbicsJzEnLHInbWludXRlcz98bWlucz8nLHIn5YiG6ZCYJyksKCdkdXJhdGlvbicsJzYwJyxyJ2hvdXJzP3xocnM/JyxyJ+Wwj+aZgicpLAogKCdkYXlzJywnMScscidkYXlzPycsciflpKl85pelJyksKCd3ZWVrcycsJzEnLHInd2Vla3M/JyxyJ+aYn+acn3zpgLF85ZGoJyksCiAoJ2FnZScsJzEnLHIneWVhcnM/IG9sZHx5ZWFycz8gb2YgYWdlJyxyJ+atsicpLAogKCdhbW91bnQgSEtEJywnMScscidIS0R8SEsgZG9sbGFycz98SG9uZyBLb25nIGRvbGxhcnM/JyxyJ+a4r+WFgycpLAogKCdwZXJjZW50YWdlJywnMScsciclfHBlcmNlbnR8cGVyIGNlbnQnLHInJXzlt7Tku5knKSwKICgncGFnZXMnLCcxJyxyJ3BhZ2VzPycscifpoIEnKSwoJ2Zsb29ycycsJzEnLHInZmxvb3JzPycscifmqJMnKSwKICgnb2NjdXJyZW5jZXMnLCcxJyxyJ3RpbWVzPycscifmrKEnKSwKICgnY291bnQnLCcxJyxyJ3Blb3BsZXxwZXJzb25zP3xwdXBpbHM/fHN0dWRlbnRzP3xwYXJ0aWNpcGFudHM/fGNvcGllc3xpdGVtcz98Zm9ybXM/JyxyJ+S6unzlkI185YCLfOS7vScpLApdCk1PTlRIUz17eDppKzEgZm9yIGkseCBpbiBlbnVtZXJhdGUoJ0phbnVhcnkgRmVicnVhcnkgTWFyY2ggQXByaWwgTWF5IEp1bmUgSnVseSBBdWd1c3QgU2VwdGVtYmVyIE9jdG9iZXIgTm92ZW1iZXIgRGVjZW1iZXInLnNwbGl0KCkpfQoKZGVmIHF1YW50aXR5X2ZhY3RzKHRleHQsIGxhbmcpOgogICAgcmVzdWx0PWRlZmF1bHRkaWN0KGxpc3QpCiAgICB0PW5vcm1hbCh0ZXh0KQogICAgbnVtPU5VTSBpZiBsYW5nPT0nemgnIGVsc2UgcicoPzpcZCsoPzpbLC5dXGQrKSp8JysgJ3wnLmpvaW4oRU5OVU0pK3InKScKICAgIGZvciBkaW1lbnNpb24sbXVsdCxlcCx6cCBpbiBVTklUUzoKICAgICAgICBwYXQ9KHJmJyh7bnVtfSlccyooPzp7enB9KSg/IVthLXpdKScgaWYgbGFuZz09J3poJyBlbHNlIHJmJ1xiKHtudW19KVxzKig/OntlcH0pKD8hW2Etel0pJykKICAgICAgICBmb3IgbSBpbiByZS5maW5kaXRlcihwYXQsdCxyZS5JKToKICAgICAgICAgICAgIyDkuozmrKEgLyDkuIDoiKwgLyDnmb7liIbmr5QgY2Fubm90IGJlIG1pc3Rha2VuIGZvciBudW1lcmljIHF1YW50aXRpZXMuCiAgICAgICAgICAgIHJlc3VsdFtkaW1lbnNpb25dLmFwcGVuZCgoc3RyKG51bWJlcihtWzFdKSpEZWNpbWFsKG11bHQpKSxtWzBdKSkKICAgIGlmIGxhbmc9PSdlbic6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYkhLXCRccyooXGQrKD86WywuXVxkKykqKScsdCxyZS5JKTpyZXN1bHRbJ2Ftb3VudCBIS0QnXS5hcHBlbmQoKHN0cihudW1iZXIobVsxXSkpLG1bMF0pKQogICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHInXGIoXGR7MSwyfSkoPzpzdHxuZHxyZHx0aCk/XHMrKCcrJ3wnLmpvaW4oTU9OVEhTKStyJylcYicsdCxyZS5JKTpyZXN1bHRbJ2RhdGUnXS5hcHBlbmQoKGYne01PTlRIU1ttWzJdLnRpdGxlKCldfS17aW50KG1bMV0pfScsbVswXSkpCiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYignKyd8Jy5qb2luKE1PTlRIUykrcicpXHMrKFxkezEsMn0pKD86c3R8bmR8cmR8dGgpP1xiJyx0LHJlLkkpOnJlc3VsdFsnZGF0ZSddLmFwcGVuZCgoZid7TU9OVEhTW21bMV0udGl0bGUoKV19LXtpbnQobVsyXSl9JyxtWzBdKSkKICAgIGVsc2U6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocmYnKHtOVU19KeaciCh7TlVNfSlb5pel6JmfXScsdCk6cmVzdWx0WydkYXRlJ10uYXBwZW5kKChmJ3tudW1iZXIobVsxXSl9LXtudW1iZXIobVsyXSl9JyxtWzBdKSkKICAgICAgICAjIERhdGVzIGFyZSBub3QgZHVyYXRpb25zLiBBIGRhdGUncyDml6UgaXMgcmVtb3ZlZCBmcm9tIGRheSBxdWFudGl0aWVzLgogICAgICAgIHJlc3VsdFsnZGF5cyddPVtxIGZvciBxIGluIHJlc3VsdFsnZGF5cyddIGlmIG5vdCByZS5zZWFyY2gocifmnIgnK3JlLmVzY2FwZShxWzFdKSx0KV0KICAgICMgRGVjaW1hbHMgYXJlIGNhbm9uaWNhbGlzZWQgZm9yIGVxdWFsaXR5IGFmdGVyIFNJIGNvbnZlcnNpb24uCiAgICBmb3IgY2F0ZWdvcnksIGhpdHMgaW4gcmVzdWx0Lml0ZW1zKCk6CiAgICAgICAgaWYgY2F0ZWdvcnkhPSdkYXRlJzpyZXN1bHRbY2F0ZWdvcnldPVsoc3RyKERlY2ltYWwodikubm9ybWFsaXplKCkpLHMpIGZvciB2LHMgaW4gaGl0c10KICAgIHJldHVybiBkaWN0KHJlc3VsdCkKCgpkZWYgZGlyZWN0X2V2aWRlbmNlKGNoaW5lc2UsIGVuZ2xpc2gpOgogICAgeixlPW5vcm1hbChjaGluZXNlKSxub3JtYWwoZW5nbGlzaCkKICAgIG91dD1bXQogICAgZm9yIGV4dHJhY3RvciBpbiAoY29uY2VwdF9mYWN0cyxxdWFudGl0eV9mYWN0cyk6CiAgICAgICAgemYsZWY9ZXh0cmFjdG9yKHosJ3poJyksZXh0cmFjdG9yKGUsJ2VuJykKICAgICAgICBmb3IgY2F0ZWdvcnkgaW4gc29ydGVkKHpmLmtleXMoKSAmIGVmLmtleXMoKSk6CiAgICAgICAgICAgIGEsYj16ZltjYXRlZ29yeV0sZWZbY2F0ZWdvcnldCiAgICAgICAgICAgIGlmIGV4dHJhY3Rvcj09cXVhbnRpdHlfZmFjdHM6CiAgICAgICAgICAgICAgICAjIFNldHMgb2YgdmFsdWVzIGNhbm5vdCBhc3NpZ24gc2VtYW50aWMgcm9sZXM7IGRlZmVyIHJlcGVhdGVkCiAgICAgICAgICAgICAgICAjIGRpc3RpbmN0IHF1YW50aXRpZXMgdG8gY29udGV4dHVhbCBpbmZlcmVuY2UgcmF0aGVyIHRoYW4gZ3Vlc3MuCiAgICAgICAgICAgICAgICBhLGI9dW5pcXVlKGEpLHVuaXF1ZShiKQogICAgICAgICAgICBpZiBhIGFuZCBiIGFuZCBhWzBdIT1iWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoY2F0ZWdvcnksYSxiKSkKICAgIGZvciBjYXRlZ29yeSxwYXR0ZXJuIGluIFsKICAgICAgICAoJ2VtYWlsIGFkZHJlc3MnLHInW1x3ListXStAW1x3Li1dK1wuW2EtekEtWl17Mix9JyksCiAgICAgICAgKCdVUkwgZGVzdGluYXRpb24nLHInaHR0cHM/Oi8vW15cczw+77yM44CC77yb77yJ44CNXSsnKSwKICAgICAgICAoJ3JlZmVyZW5jZSBpZGVudGlmaWVyJyxyJyg/PCFbQS1aYS16XGRdKVtBLVpdezEsNX1bLS9dXGR7Mix9KD86Wy0vXVxkKyk/JyldOgogICAgICAgIHpzPVsobVswXS5yc3RyaXAoJy4sOyknKSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXR0ZXJuLHopXQogICAgICAgIGVzPVsobVswXS5yc3RyaXAoJy4sOyknKSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXR0ZXJuLGUpXQogICAgICAgIGEsYj11bmlxdWUoenMpLHVuaXF1ZShlcykKICAgICAgICBpZiBhIGFuZCBiIGFuZCBhWzBdIT1iWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoY2F0ZWdvcnksYSxiKSkKICAgIHpjLGVjPWNsb2NrX2ZhY3RzKHosJ3poJyksY2xvY2tfZmFjdHMoZSwnZW4nKQogICAgYSxiPXVuaXF1ZSh6YyksdW5pcXVlKGVjKQogICAgaWYgYSBhbmQgYiBhbmQgYVswXSE9YlswXTpvdXQuYXBwZW5kKGV2aWRlbmNlKCdjbG9jayB0aW1lJyxhLGIpKQogICAgb3V0LmV4dGVuZChib3VuZGFyeV9ldmlkZW5jZSh6LGUpKQogICAgb3V0LmV4dGVuZChtb2RhbGl0eV9ldmlkZW5jZSh6LGUpKQogICAgcmV0dXJuIG91dAoKCmRlZiBib3VuZGFyeV9ldmlkZW5jZSh6LGUpOgogICAgb3V0PVtdCiAgICAjIFBvc2l0aW9uIHdvcmRzIG9ubHkgd2hlbiB1c2VkIHNwYXRpYWxseSwgbm90IHRoZSBwYXN0IHRlbnNlIG9mIGxlYXZlLgogICAgZW5fZGlyZWN0aW9ucz1bKHYscicoPzooPzp0b3x0b3dhcmRzP3xwb2ludHM/fHBvaW50aW5nfGZhY2luZ3x0dXJuKVxzKyg/OnRoZVxzKyk/JytwYXQrcid8JytwYXQrcidbLSBdaGFuZCknKSBmb3IgdixwYXQgaW4gWygnbGVmdCcsJ2xlZnQnKSwoJ3JpZ2h0JywncmlnaHQnKSwoJ3VwJywndXB3YXJkcz98dXAnKSwoJ2Rvd24nLCdkb3dud2FyZHM/fGRvd24nKV1dCiAgICB6aF9kaXJlY3Rpb25zPVsoJ2xlZnQnLHIn5bemKD865pa5fOWBtHzpgoopJyksKCdyaWdodCcsciflj7MoPzrmlrl85YG0fOmCiiknKSwoJ3VwJyxyJ+S4iig/OuaWuXzlgbR86YKKKScpLCgnZG93bicscifkuIsoPzrmlrl85YG0fOmCiiknKV0KICAgIHpoPXVuaXF1ZShbKHYsbVswXSkgZm9yIHYscCBpbiB6aF9kaXJlY3Rpb25zIGZvciBtIGluIHJlLmZpbmRpdGVyKHAseildKQogICAgZW49dW5pcXVlKFsodixtWzBdKSBmb3IgdixwIGluIGVuX2RpcmVjdGlvbnMgZm9yIG0gaW4gcmUuZmluZGl0ZXIocCxlLHJlLkkpXSkKICAgIGlmIHpoIGFuZCBlbiBhbmQgemhbMF0hPWVuWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoJ3NwYXRpYWwgZGlyZWN0aW9uJyx6aCxlbikpCiAgICAjIFVuaXQtb25seSBpbnN0cnVjdGlvbnMgY2FuIGNoYW5nZSBldmVuIHdoZW4gdGhlcmUgaXMgbm8gbnVtZXJpYyB2YWx1ZS4KICAgIGZvciBkaW0gaW4gWydsZW5ndGgnLCdtYXNzJ106CiAgICAgICAgenM9W107ZXM9W10KICAgICAgICBmb3IgZCxtdWx0LGVwLHpwIGluIFVOSVRTOgogICAgICAgICAgICBpZiBkIT1kaW06Y29udGludWUKICAgICAgICAgICAgenMuZXh0ZW5kKChtdWx0LG1bMF0pIGZvciBtIGluIHJlLmZpbmRpdGVyKHInKD865LulfOeUqHzllq7kvY1b54K65pivOu+8ml0pKD86Jyt6cCtyJyknLHopKQogICAgICAgICAgICBlcy5leHRlbmQoKG11bHQsbVswXSkgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYig/OmlufHVzaW5nKVxzKyg/OicrZXArcicpXGInLGUscmUuSSkpCiAgICAgICAgYSxiPXVuaXF1ZSh6cyksdW5pcXVlKGVzKQogICAgICAgIGlmIGEgYW5kIGIgYW5kIGFbMF0hPWJbMF06b3V0LmFwcGVuZChldmlkZW5jZShkaW0rJyB1bml0JyxhLGIpKQogICAgIyBJbmNsdXNpdmUgLyBleGNsdXNpdmUgYm91bmRhcmllcyBtdXN0IHJlZmVyIHRvIHRoZSBzYW1lIGV4cGxpY2l0IHZhbHVlLgogICAgem49TlVNO2VuPXInXGQrKD86XC5cZCspPycKICAgIHpocGF0dGVybnM9WygnPj0nLHJmJ+S4jeWwkeaWvCh7em59KXzmnIDlsJEoe3pufSl8KHt6bn0pKD865q2yKT/miJbku6XkuIonKSwoJz4nLHJmJ+i2hemBjih7em59KXzpq5jmlrwoe3pufSknKSwoJzw9JyxyZifkuI3lpJrmlrwoe3pufSl85pyA5aSaKHt6bn0pfCh7em59KSg/Ouatsik/5oiW5Lul5LiLJyksKCc8JyxyZiflsJHmlrwoe3pufSl85pyq5ru/KHt6bn0pfOS9juaWvCh7em59KScpXQogICAgZW5wYXR0ZXJucz1bKCc+PScscmYnXGJhdCBsZWFzdCAoe2VufSl8XGIoe2VufSkoPzogeWVhcnM/KT8gb3IgKD86bW9yZXxvdmVyfGFib3ZlKScpLCgnPicscmYnXGJtb3JlIHRoYW4gKHtlbn0pfFxiYWJvdmUgKHtlbn0pJyksKCc8PScscmYnXGJhdCBtb3N0ICh7ZW59KXxcYm5vIG1vcmUgdGhhbiAoe2VufSl8XGIoe2VufSkoPzogeWVhcnM/KT8gb3IgKD86bGVzc3x1bmRlcnxiZWxvdyknKSwoJzwnLHJmJ1xibGVzcyB0aGFuICh7ZW59KXxcYnVuZGVyICh7ZW59KXxcYmJlbG93ICh7ZW59KScpXQogICAgZGVmIGxpbWl0cyhwcyx0KToKICAgICAgICBmb3VuZD1bXQogICAgICAgIGZvciBvcCxwYXQgaW4gcHM6CiAgICAgICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHBhdCx0LHJlLkkpOmZvdW5kLmFwcGVuZCgoKG9wLHN0cihudW1iZXIobmV4dChnIGZvciBnIGluIG0uZ3JvdXBzKCkgaWYgZykpKSksbVswXSkpCiAgICAgICAgcmV0dXJuIHVuaXF1ZShmb3VuZCkKICAgIGEsYj1saW1pdHMoemhwYXR0ZXJucyx6KSxsaW1pdHMoZW5wYXR0ZXJucyxlKQogICAgaWYgYSBhbmQgYiBhbmQgYVswXVsxXT09YlswXVsxXSBhbmQgYVswXVswXSE9YlswXVswXTpvdXQuYXBwZW5kKGV2aWRlbmNlKCdpbmNsdXNpdmUvZXhjbHVzaXZlIGJvdW5kYXJ5JyxhLGIpKQogICAgcmV0dXJuIG91dAoKIyBPcHRpb25hbCB2ZXJzdXMgcmVxdWlyZWQgaXMgbWF0ZXJpYWwgZXZlbiB3aGVuIG5laXRoZXIgc2lkZSBpcyBhIGxvZ2ljYWwKIyBjb250cmFkaWN0aW9uLiBPbmx5IGF0dGFjaCBhbiBleHBsaWNpdCBtb2RhbCBpbW1lZGlhdGVseSBiZWZvcmUgYSBzaGFyZWQgYWN0aW9uLgpBQ1RJT05TID0gewogJ3N1Ym1pdCc6IChyJ3N1Ym1pdHxsb2RnZXxoYW5kIGluJyxyJ+aPkOS6pHzpgZ7kuqR85Lqk5ZueJyksCiAnYnJpbmcnOiAocidicmluZycscifmlJzluLZ85pSc5ZCMfOW4tuWCmXzluLblkIwnKSwKICdzaG93JzogKHInc2hvd3xwcmVzZW50fHByb2R1Y2UnLHIn5Ye656S6JyksCiAncGF5JzogKHIncGF5JyxyJ+e5s+S7mHzmlK/ku5h857mz5LqkJyksCiAncmVnaXN0ZXInOiAocidyZWdpc3RlcnxlbnJvbHxlbnJvbGwnLHIn55m76KiYfOWgseWQjScpLAogJ2F0dGVuZCc6IChyJ2F0dGVuZCcsciflh7rluK0nKSwKICd1cGxvYWQnOiAocid1cGxvYWQnLHIn5LiK6LyJfOS4iuWCsycpLAogJ3NpZ24nOiAocidzaWduJyxyJ+ewvee9sicpLAogJ3JldGFpbic6IChyJ2tlZXB8cmV0YWluJyxyJ+S/neeVmScpLAogJ2NvbGxlY3QnOiAocidjb2xsZWN0JyxyJ+mgmOWPlnzlj5bku7YnKSwKICdyZXBseSc6IChyJ3JlcGx5fHJlc3BvbmQnLHIn5Zue6KaGfOWbnuW+qScpLAogJ3NlbmQnOiAocidzZW5kfGZvcndhcmQnLHIn55m86YCBfOWvhOmAgScpLAogJ3VzZSc6IChyJ3VzZScscifkvb/nlKgnKSwKICdlbnRlcic6IChyJ2VudGVyJyxyJ+mAsuWFpScpLAogJ2NhbmNlbCc6IChyJ2NhbmNlbCcsciflj5bmtognKSwKICdyZXBsYWNlJzogKHIncmVwbGFjZScscifmm7Tmj5t85Y+W5LujJyksCn0KCmRlZiBtb2RhbGl0eV9ldmlkZW5jZSh6LGUpOgogICAgIyBVbnJlc29sdmVkIGNvbmRpdGlvbmFsIHNjb3BlIGlzIGxlZnQgdG8gY29udGV4dHVhbCBpbmZlcmVuY2UsIG5vdCByZWR1Y2VkCiAgICAjIHRvIGEgZG9jdW1lbnQtd2lkZSBiYWcgb2Ygb2JsaWdhdGlvbiB0b2tlbnMuCiAgICBpZiByZS5zZWFyY2gocidcYig/OmlmfHVubGVzc3xvbmx5fGV4Y2VwdHx3aGV0aGVyKVxiJyxlLHJlLkkpIG9yIHJlLnNlYXJjaChyJ+Wmgnzoi6V86Zmk6Z2efOWPquaciXzlg4V85omNfOmZpOWklicseik6cmV0dXJuIFtdCiAgICBvdXQ9W10KICAgIGVtPVsoJ29wdGlvbmFsJyxyJyg/Om5lZWQgbm90fGRvKD86ZXMpPyBub3QgKD86bmVlZHxoYXZlKSB0b3woPzppc3xhcmUpIG5vdCByZXF1aXJlZCB0bylccyokJyksCiAgICAgICAgKCdwcm9oaWJpdGVkJyxyJyg/Om11c3Qgbm90fG1heSBub3R8Y2Fubm90fGNhblx4Mjd0fCg/OmlzfGFyZSkgbm90IGFsbG93ZWQgdG8pXHMqJCcpLAogICAgICAgICgncmVxdWlyZWQnLHInKD88IW5vdCApKD86bXVzdHxuZWVkIHRvfGhhdmUgdG98aGFzIHRvfCg/OmlzfGFyZSkgcmVxdWlyZWQgdG8pXHMqJCcpLAogICAgICAgICgncGVybWl0dGVkJyxyJyg/Om1heXxjYW58KD86aXN8YXJlKSBhbGxvd2VkIHRvKVxzKiQnKV0KICAgIHptPVsoJ29wdGlvbmFsJyxyJyg/OueEoemgiHzmr4vpoIh85LiN5b+FfOS4jeeUqClccyokJyksKCdwcm9oaWJpdGVkJyxyJyg/OuS4jeW+l3zkuI3lj69856aB5q2ifOS4jeWHhilccyokJyksCiAgICAgICAgKCdyZXF1aXJlZCcscicoPzwhW+avi+eEoeS4jV0pKD865b+F6aCIfOmgiClccyokJyksKCdwZXJtaXR0ZWQnLHInKD88IVvkuI1dKSg/OuWPr+S7pXzlj68pXHMqJCcpXQogICAgZGVmIGZhY3QodCxhcCxwYXR0ZXJucyxsYW5nKToKICAgICAgICBhY3Rpb25zPWxpc3QocmUuZmluZGl0ZXIocidcYig/OicrYXArcicpXGInIGlmIGxhbmc9PSdlbicgZWxzZSBhcCx0LHJlLkkpKQogICAgICAgIGlmIGxlbihhY3Rpb25zKSE9MTpyZXR1cm4gTm9uZQogICAgICAgIGE9YWN0aW9uc1swXTtwcmVmaXg9dFttYXgoMCxhLnN0YXJ0KCktNjApOmEuc3RhcnQoKV0KICAgICAgICBmb3IgdmFsdWUscGF0IGluIHBhdHRlcm5zOgogICAgICAgICAgICBtPXJlLnNlYXJjaChwYXQscHJlZml4LHJlLkkpCiAgICAgICAgICAgIGlmIG06cmV0dXJuIHZhbHVlLG1bMF0rYVswXQogICAgICAgIHJldHVybiBOb25lCiAgICBmb3IgbmFtZSwoZXAsenApIGluIEFDVElPTlMuaXRlbXMoKToKICAgICAgICBhLGI9ZmFjdCh6LHpwLHptLCd6aCcpLGZhY3QoZSxlcCxlbSwnZW4nKQogICAgICAgICMgT3B0aW9uYWwgdnMgcGVybWl0dGVkIGNhbiBiZSBlcXVpdmFsZW50OyBuZWl0aGVyIGVzdGFibGlzaGVzIGFuIG9ibGlnYXRpb24uCiAgICAgICAgaWYgYSBhbmQgYiBhbmQgYVswXSE9YlswXSBhbmQgbm90IHthWzBdLGJbMF19PD17J29wdGlvbmFsJywncGVybWl0dGVkJ306CiAgICAgICAgICAgIG91dC5hcHBlbmQoZXZpZGVuY2UoJ29ibGlnYXRpb24gdG8gJytuYW1lLGEsYikpCiAgICByZXR1cm4gb3V0CgoKZGVmIGNsb2NrX2ZhY3RzKHRleHQsbGFuZyk6CiAgICBmb3VuZD1bXTtvY2N1cGllZD1bXQogICAgaWYgbGFuZz09J2VuJzoKICAgICAgICBmb3IgbSBpbiByZS5maW5kaXRlcihyJ1xiKFxkezEsMn0pKD86WzouXShcZHsyfSkpP1xzKihbYXBdKVwuP21cLj8nLHRleHQscmUuSSk6CiAgICAgICAgICAgIGgsbW49aW50KG1bMV0pLGludChtWzJdIG9yIDApCiAgICAgICAgICAgIGlmIDE8PWg8PTEyIGFuZCBtbjw2MDoKICAgICAgICAgICAgICAgIGZvdW5kLmFwcGVuZCgoc3RyKChoJTEyKygxMiBpZiBtWzNdLmxvd2VyKCk9PSdwJyBlbHNlIDApKSo2MCttbiksbVswXSkpO29jY3VwaWVkLmFwcGVuZChtLnNwYW4oKSkKICAgICAgICBmb3IgdmFsLHAgaW4gWyg3MjAscidcYm5vb25cYicpLCgwLHInXGJtaWRuaWdodFxiJyldOgogICAgICAgICAgICBmb3VuZC5leHRlbmQoKHN0cih2YWwpLG1bMF0pIGZvciBtIGluIHJlLmZpbmRpdGVyKHAsdGV4dCxyZS5JKSkKICAgIGVsc2U6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocmYnKOS4iuWNiHzkuIvljYh85pma5LiKfOWHjOaZqHzkuK3ljYgpKHtOVU19KSg/OuaZgnzpu558OikoPzooe05VTX0p5YiGPyk/Jyx0ZXh0KToKICAgICAgICAgICAgaCxtbj1pbnQobnVtYmVyKG1bMl0pKSxpbnQobnVtYmVyKG1bM10pKSBpZiBtWzNdIGVsc2UgMAogICAgICAgICAgICBpZiAwPD1oPD0xMiBhbmQgbW48NjA6CiAgICAgICAgICAgICAgICBmb3VuZC5hcHBlbmQoKHN0cigoaCUxMisoMTIgaWYgbVsxXSBpbiBbJ+S4i+WNiCcsJ+aZmuS4iicsJ+S4reWNiCddIGVsc2UgMCkpKjYwK21uKSxtWzBdKSk7b2NjdXBpZWQuYXBwZW5kKG0uc3BhbigpKQogICAgICAgIGZvciB2YWwscCBpbiBbKDcyMCxyJ+ato+WNiCcpLCgwLHIn5Y2I5aScJyldOmZvdW5kLmV4dGVuZCgoc3RyKHZhbCksbVswXSkgZm9yIG0gaW4gcmUuZmluZGl0ZXIocCx0ZXh0KSkKICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHInKD88IVxkKShcZHsxLDJ9KTooXGR7Mn0pKD8hXGQpJyx0ZXh0KToKICAgICAgICBpZiBhbnkobS5zdGFydCgpPGIgYW5kIG0uZW5kKCk+YSBmb3IgYSxiIGluIG9jY3VwaWVkKTpjb250aW51ZQogICAgICAgIGgsbW49aW50KG1bMV0pLGludChtWzJdKQogICAgICAgIGlmIGg8MjQgYW5kIG1uPDYwOmZvdW5kLmFwcGVuZCgoc3RyKGgqNjArbW4pLG1bMF0pKQogICAgcmV0dXJuIGZvdW5kCg==', 'terminology.py': 'IiIiQm91bmRlZCB0ZXJtaW5vbG9neSBmb3IgdHJhbnNsYXRpb24gY3Jvc3MtY2hlY2tzLCBpbmRlcGVuZGVudCBvZiBub3RpY2Ugd29yZGluZy4KClRoaXMgaXMgYSBnbG9zc2FyeSwgbm90IGFuIGVxdWl2YWxlbmNlIHZlcmRpY3Qgb3IgYSBzdXBwb3J0ZWQtbm90aWNlIHdoaXRlbGlzdC4KVGVybXMgbm90IGxpc3RlZCBjb250aW51ZSB0aHJvdWdoIG5vcm1hbCBjb250ZXh0dWFsIHRyYW5zbGF0aW9uLiBObyBiZW5jaG1hcmsKaWRlbnRpZmllcnMsIHRhcmdldCB3b3JkaW5nLCBsYWJlbHMgb3IgZ29sZCByYXRpb25hbGVzIGVudGVyIHRoaXMgc3RlcC4KIiIiCmltcG9ydCByZQpmcm9tIG9wZW5jYyBpbXBvcnQgT3BlbkNDCl9UT19TSU1QTElGSUVEPU9wZW5DQygidDJzIikKClRFUk1TID0gewogICAgJ+amleaouSc6J2JhbnlhbiB0cmVlJywgJ+aon+aouSc6J2NhbXBob3IgdHJlZScsICfmtIvntKvojYonOidvcmNoaWQgdHJlZScsCiAgICAn55u45oCd5qi5JzonYWNhY2lhIHRyZWUnLCAn5p2+5qi5JzoncGluZSB0cmVlJywgJ+apoeaouSc6J29hayB0cmVlJywKICAgICfmpZPmqLknOidtYXBsZSB0cmVlJywgJ+ajlearmuaouSc6J3BhbG0gdHJlZScsICfmnKjmo4nmqLknOidjb3R0b24gdHJlZScsCiAgICAn5p+z5qi5Jzond2lsbG93IHRyZWUnLCAn6YqA5p2P5qi5JzonZ2lua2dvIHRyZWUnLAogICAgJ+ijneijsSc6J21vdW50aW5nIGFydHdvcmsnLAogICAgJ+ebtOWQkeagvOW8jyc6J3BvcnRyYWl0IGZvcm1hdCcsICfnuLHlkJHmoLzlvI8nOidwb3J0cmFpdCBmb3JtYXQnLAogICAgJ+apq+WQkeagvOW8jyc6J2xhbmRzY2FwZSBmb3JtYXQnLAogICAgJ+avq+exsyc6J21pbGxpbWV0cmVzJywgJ+WOmOexsyc6J2NlbnRpbWV0cmVzJywgJ+WFrOaWpCc6J2tpbG9ncmFtcycsCiAgICAn5riv5YWDJzonSG9uZyBLb25nIGRvbGxhcnMnLCAn5bel5L2c5aSpJzond29ya2luZyBkYXlzJywgJ+abhuaXpSc6J2NhbGVuZGFyIGRheXMnLAogICAgJ+ato+acrCc6J29yaWdpbmFsIGRvY3VtZW50JywgJ+W9seWNsOacrCc6J3Bob3RvY29weScsCiAgICAn5qWt5Li756uL5qGI5rOV5ZyYJzoib3duZXJzJyBjb3Jwb3JhdGlvbiIsICfmpa3kuLvms5XlnJgnOiJvd25lcnMnIGNvcnBvcmF0aW9uIiwKICAgICfpnZ7niZ/liKknOidub25wcm9maXQnLCAn56eB6ZqxJzoncHJpdmFjeScsICfpm7vpg7UnOidlbWFpbCcsCn0KX1BBVFRFUk49cmUuY29tcGlsZSgnfCcuam9pbihtYXAocmUuZXNjYXBlLHNvcnRlZChURVJNUyxrZXk9bGVuLHJldmVyc2U9VHJ1ZSkpKSkKCmRlZiB0cmFuc2xhdGlvbl9zb3VyY2UoY2hpbmVzZSk6CiAgICAjIEFuY2hvciBrbm93biB0ZWNobmljYWwgdGVybXMgYmVmb3JlIGdlbmVyYXRpb24sIHNvIGEgbWlzdHJhbnNsYXRlZCByYXJlCiAgICAjIHdvcmQgZG9lcyBub3QgbWFzcXVlcmFkZSBhcyBkaXNhZ3JlZW1lbnQgd2l0aCB0aGUgc3VwcGxpZWQgRW5nbGlzaC4KICAgIHJldHVybiBfVE9fU0lNUExJRklFRC5jb252ZXJ0KF9QQVRURVJOLnN1YihsYW1iZGEgbTonICcrVEVSTVNbbVswXV0rJyAnLGNoaW5lc2UpKQo=', 'config.json': 'ewogICJtdF9iaWRpcmVjdGlvbmFsIjogMC43NSwKICAibXRfb25ld2F5IjogMC43NSwKICAiY3Jvc3Nfc3VwcG9ydCI6IDAuODUsCiAgInNpbWlsYXJpdHlfbWF4IjogMC42LAogICJtdF9sb3dfc2ltaWxhcml0eSI6IDAuNzUsCiAgImZhY3RfYWxpZ25tZW50X21pbiI6IDAuNiwKICAidHJhbnNsYXRpb25fcXVhbGl0eV9taW4iOiAwLjU1LAogICJzZW1hbnRpY19hbGlnbm1lbnRfbWluIjogMC40NSwKICAid2hvbGVfY3Jvc3NfbWluIjogMC45LAogICJ3aG9sZV9tdF9taW4iOiAwLjk1Cn0K', 'run.py': 'IiIiUnVuIHRoZSBmcm96ZW4gcHJlZGljdG9yIG9uIGFuIGlucHV0cy1vbmx5IENTVjsgbmV2ZXIgc2NvcmUgZHVyaW5nIGluZmVyZW5jZS4iIiIKaW1wb3J0IGFyZ3BhcnNlLGNzdixqc29uLGhhc2hsaWIsc3lzLHRpbWUscGxhdGZvcm0saW1wb3J0bGliLm1ldGFkYXRhLGlvCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSx0aW1lem9uZQpmcm9tIGVuZ2luZSBpbXBvcnQgQ2hlY2tlcixNT0RFTF9JRFMKcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCnAuYWRkX2FyZ3VtZW50KCctLWlucHV0cycscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1jYWNoZScscmVxdWlyZWQ9VHJ1ZSkKcC5hZGRfYXJndW1lbnQoJy0tYmF0Y2gtc2l6ZScsdHlwZT1pbnQsZGVmYXVsdD01MCk7cC5hZGRfYXJndW1lbnQoJy0tcmVxdWlyZS1mcm96ZW4nLGFjdGlvbj0nc3RvcmVfdHJ1ZScpCmE9cC5wYXJzZV9hcmdzKCk7cm9vdD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnNvdXJjZV9maWxlcz1bJ2VuZ2luZS5weScsJ2ZhY3RzLnB5JywndGVybWlub2xvZ3kucHknLCdjb25maWcuanNvbicsJ3J1bi5weSddCmhhc2hlcz17ZjpoYXNobGliLnNoYTI1Nigocm9vdC9mKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpIGZvciBmIGluIHNvdXJjZV9maWxlc30KaWYgYS5yZXF1aXJlX2Zyb3plbjoKIG1hbmlmZXN0PWpzb24ubG9hZHMoKHJvb3QvJ2Zyb3plbl9tYW5pZmVzdC5qc29uJykucmVhZF90ZXh0KCkpO2Fzc2VydCBoYXNoZXM9PW1hbmlmZXN0WydmaWxlcyddLCdGcm96ZW4gY2hlY2tlciBjaGFuZ2VkJwpyYXc9UGF0aChhLmlucHV0cykucmVhZF9ieXRlcygpCnJvd3M9bGlzdChjc3YuRGljdFJlYWRlcihpby5TdHJpbmdJTyhyYXcuZGVjb2RlKCd1dGYtOC1zaWcnKSkpKQphc3NlcnQgc2V0KHJvd3NbMF0pPT17J2Nhc2VfaWQnLCdlbmdsaXNoJywnY2hpbmVzZSd9LCdQcmVkaWN0aW9uIGlucHV0IG11c3QgY29udGFpbiBubyBnb2xkIGxhYmVscyBvciByYXRpb25hbGVzJwphc3NlcnQgbGVuKHtyWydjYXNlX2lkJ10gZm9yIHIgaW4gcm93c30pPT1sZW4ocm93cykKc3RhcnRlZD1kYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKQpjaGVja2VyPUNoZWNrZXIoY2FjaGU9YS5jYWNoZSkKcmVzdWx0cz1bXTtvdXRwdXQ9UGF0aChhLm91dHB1dCk7b3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKcGFydGlhbD1vdXRwdXQud2l0aF9zdWZmaXgoJy5qc29ubCcpO3BhcnRpYWwud3JpdGVfdGV4dCgnJykKZm9yIGkgaW4gcmFuZ2UoMCxsZW4ocm93cyksYS5iYXRjaF9zaXplKToKIGJhdGNoPWNoZWNrZXIucHJlZGljdF9tYW55KHJvd3NbaTppK2EuYmF0Y2hfc2l6ZV0pO3Jlc3VsdHMuZXh0ZW5kKGJhdGNoKQogd2l0aCBwYXJ0aWFsLm9wZW4oJ2EnKSBhcyBmOgogIGZvciByIGluIGJhdGNoOmYud3JpdGUoanNvbi5kdW1wcyhyLGVuc3VyZV9hc2NpaT1GYWxzZSkrJ1xuJykKIHByaW50KCdQUkVESUNUSU9OUycsbGVuKHJlc3VsdHMpLCcvJyxsZW4ocm93cyksZmx1c2g9VHJ1ZSkKbWV0YWRhdGE9ZGljdCh2ZXJzaW9uPSdWMTAnLHN0YXJ0ZWQ9c3RhcnRlZCxjb21wbGV0ZWQ9ZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiBpbnB1dHNfc2hhMjU2PWhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCksZmlsZXM9aGFzaGVzLG1vZGVsX2lkcz1NT0RFTF9JRFMsCiBtb2RlbF9yZXZpc2lvbnM9Y2hlY2tlci5yZXZpc2lvbnMsZGV2aWNlPWNoZWNrZXIuZGV2aWNlLHB5dGhvbj1wbGF0Zm9ybS5weXRob25fdmVyc2lvbigpLAogcGFja2FnZXM9e3A6aW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24ocCkgZm9yIHAgaW4gWyd0b3JjaCcsJ3RyYW5zZm9ybWVycycsJ3NlbnRlbmNlLXRyYW5zZm9ybWVycycsJ3NlbnRlbmNlcGllY2UnLCdwYW5kYXMnLCdzY2lweScsJ251bXB5Jywnb3BlbmNjLXB5dGhvbi1yZWltcGxlbWVudGVkJ119LAogcHJlZGljdGlvbnM9bGVuKHJlc3VsdHMpLHY3X2ZhbGxiYWNrX2NhbGxzPTApCm91dHB1dC53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChtZXRhZGF0YT1tZXRhZGF0YSxyZXN1bHRzPXJlc3VsdHMpLGVuc3VyZV9hc2NpaT1GYWxzZSxpbmRlbnQ9MikpCnByaW50KCdWMTAgQ09NUExFVEUnLGxlbihyZXN1bHRzKSxmbHVzaD1UcnVlKQo=', 'frozen_manifest.json': 'ewogICJ2ZXJzaW9uIjogIlYxMCIsCiAgImZyb3plbl9hdCI6ICIyMDI2LTExLTE1VDAxOjQ1OjA4LjU2MzgyOCswMDowMCIsCiAgImZpbGVzIjogewogICAgImVuZ2luZS5weSI6ICJhM2JlZTU4NmI0NDg2NjhmNjg3NTU5YTZjYTMxZWI5NDdlYWRiZTdkOGI0NTBmZjMzZWZiOGY4ZTc0NjZiZmI0IiwKICAgICJmYWN0cy5weSI6ICJjY2JmM2EwNTc4MWZlZGE3OTc0YjJiNzk2MDAzMjEyYmM0MWNlNGU1ZmUwYmVkNTAzZWIzNDgyNTRhMjllNzhmIiwKICAgICJ0ZXJtaW5vbG9neS5weSI6ICIzZjExYTU4MTY5ZGUyYTgxZmNhZjM4NjQ3OWYzZmY4NjQxZDM5M2U4ODQ0MWE3ODEyYTNlZGNlMWZlMWRlMTkwIiwKICAgICJjb25maWcuanNvbiI6ICIxYjlkNzI3NmJlNzA0ZGM4NTEyOTFhYmEzNjdlZWQxYmY3MWM2ZmFlNzY0MDEyMWNkMTcxYzVjOWJhZWEzNWQxIiwKICAgICJydW4ucHkiOiAiNWM4ZWZmY2UwOWZjYjU0YTM3NGNlNWIzOGQ2ZGE0YzNlZjE1YjMyZTc3ZjQ1YzBkMGVmZDNlM2FiNzYwMTIyYiIKICB9LAogICJjYW5kaWRhdGVfc2hhMjU2IjogImM4ZDMxZGExNTEwYWQxZDZhMjI3MDc0MmI4NzQ1YzhjNjc2YTc5OTQxY2FkYTM3YzgyNjVmOTQ0NDNkM2Y0NDkiLAogICJjYWxpYnJhdGlvbiI6ICJPbmx5IHByaW9yIFY5IGRldmVsb3BtZW50IGNvbnRlbnQ7IGZyZXNoIGhvbGRvdXQgbm90IG9wZW5lZCBvciBldmFsdWF0ZWQiLAogICJob2xkb3V0X2lucHV0c19zaGEyNTYiOiAiNjczNjFmYzUzMmM3M2VlZjlkNTAwMGFhYjI3OTA2MTY3OTJmNDg5M2Q4YjJjZWFhY2Y0NWJlZDUyZWEwMTQ3YyIsCiAgInY3X2ZhbGxiYWNrIjogIk5vIFY3IGltcG9ydHMsIGNhbGxzLCBvciBsb2FkZWQgcnVsZXMiLAogICJ0ZXN0aW5nX2FnZW50IjogewogICAgIm1vZGVsIjogImdwdC02LjEtc29sIiwKICAgICJyZWFzb25pbmdfZWZmb3J0IjogIm1lZGl1bSIKICB9Cn0K'}
for name,data in payload.items():
    (folder/name).write_bytes(base64.b64decode(data))
manifest=json.loads((folder/'frozen_manifest.json').read_text())
for name,digest in manifest['files'].items():
    assert hashlib.sha256((folder/name).read_bytes()).hexdigest()==digest
sys.path.insert(0,str(folder))
from engine import Checker
checker=Checker(cache='/content/v10_cache.json')
check_notice=checker.check_notice
print('Frozen V10 loaded. V7 is not imported or called.')


In [ ]:
# Replace these example strings with your notice.
chinese_text='請於星期五提交表格。'
english_text='Please submit the form on Friday.'
check_notice(chinese_text,english_text)


In [ ]:
# Upload the separately supplied v10_runner_inputs.csv to /content first.
# This cell runs predictions only; ground-truth labels are not supplied to the checker.
import csv
inputs=list(csv.DictReader(open('/content/v10_runner_inputs.csv',encoding='utf-8-sig')))
assert set(inputs[0])=={'case_id','english','chinese'}
results=[]
for i in range(0,len(inputs),50):
    results.extend(checker.predict_many(inputs[i:i+50]))
    print(len(results),'/',len(inputs))
Path('/content/v10_predictions.json').write_text(json.dumps(results,ensure_ascii=False,indent=2))
print('Predictions saved. Join the answer key only after prediction is complete.')
